### 1. Setup and Data Loading



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error, mean_squared_error


# Load the data
df = pd.read_csv('survey_results_public.csv')

#Select only developers by profession and also in DACH region
df = df[(df['ConvertedCompYearly'] > 0) &
        (df['Country'].isin(['Germany', 'Austria', 'Switzerland'])) &
        (df['MainBranch'] == 'I am a developer by profession')
         ]


# Define the target variable (y) and features (X)
X = df.drop('ConvertedCompYearly', axis=1)
y = df['ConvertedCompYearly']

print("Data shape:", df.shape)
df

C:\Users\ReDI\AppData\Local\Temp\ipykernel_1364\3712056367.py:15: DtypeWarning: Columns (56,74,92,97,98,105,109,110,132,162,165) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('survey_results_public.csv')


Data shape: (2343, 149)


,ResponseId,MainBranch,Age,EdLevel,Employment,EmploymentAddl,WorkExp,LearnCodeChoose,LearnCode,LearnCodeAI,...,AIAgentOrchestration,AIAgentOrchWrite,AIAgentObserveSecure,AIAgentObsWrite,AIAgentExternal,AIAgentExtWrite,AIHuman,AIOpen,ConvertedCompYearly,JobSat
35,36,I am a developer by profession,25-34 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Employed,None of the above,10.0,"Yes, I am not new to coding but am learning ne...",AI CodeGen tools or AI-enabled apps,"Yes, I learned how to use AI-enabled tools for...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,problem solving,95132.0,6.0
49,50,I am a developer by profession,25-34 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Employed,Volunteering (regularly),6.0,"Yes, I am not new to coding but am learning ne...",Books / Physical media;Videos (not associated ...,"Yes, I learned how to use AI-enabled tools for...",...,Ollama;LangChain,NaN,NaN,NaN,ChatGPT;Claude Code;GitHub Copilot;Google Gemini,NaN,When I don’t trust AI’s answers;When I want to...,Understanding context,117793.0,8.0
50,51,I am a developer by profession,35-44 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Employed,None of the above,10.0,"No, I am not new to coding and did not learn n...",NaN,"Yes, I learned how to use AI-enabled tools for...",...,NaN,NaN,NaN,NaN,NaN,NaN,When I don’t trust AI’s answers;When I’m stuck...,"Soft skills, and the capacity to have complete...",75410.0,6.0
55,56,I am a developer by profession,18-24 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Employed,NaN,3.0,"Yes, I am not new to coding but am learning ne...","Other online resources (e.g. standard search, ...","No, I learned something that was not related t...",...,NaN,NaN,NaN,NaN,NaN,NaN,When I don’t trust AI’s answers;When I want to...,NaN,46406.0,9.0
99,100,I am a developer by profession,45-54 years old,"Secondary school (e.g. American high school, G...",Employed,None of the above,26.0,"Yes, I am not new to coding but am learning ne...",Online Courses or Certification (includes all ...,"Yes, I learned how to use AI-enabled tools for...",...,Ollama,NaN,Grafana + Prometheus,NaN,ChatGPT;Claude Code,NaN,When I don’t trust AI’s answers;When I want to...,Experience is the key when designing and devel...,145018.0,10.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48886,48887,I am a developer by profession,25-34 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Employed,Volunteering (regularly),8.0,"Yes, I am not new to coding but am learning ne...","Other online resources (e.g. standard search, ...","Yes, I learned how to use AI-enabled tools req...",...,NaN,NaN,NaN,NaN,NaN,NaN,When I don’t trust AI’s answers;When I want to...,"Thinking on your own, being able to understand...",40605.0,3.0
48927,48928,I am a developer by profession,55-64 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)","Independent contractor, freelancer, or self-em...",Engaged in paid work (20-29 hours per week),43.0,"No, I am not new to coding and did not learn n...",NaN,"Yes, I learned how to use AI-enabled tools req...",...,NaN,NaN,NaN,NaN,NaN,NaN,When I don’t trust AI’s answers;When I want to...,Analysis of user requirements.,104413.0,7.0
48953,48954,I am a developer by profession,25-34 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Employed,None of the above,5.0,"No, I am not new to coding and did not learn n...",NaN,"Yes, I learned how to use AI-enabled tools req...",...,NaN,NaN,NaN,NaN,NaN,NaN,When I don’t trust AI’s answers,Rather than knowing specific technical skills ...,75410.0,6.0
48986,48987,I am a developer by profession,35-44 years old,Some college/university study without earning ...,Employed,"Caring for dependents (children, elderly, etc.)",10.0,"Yes, I am not new to coding but am learning ne...",Online Courses or Certification (includes all ...,"Yes, I learned how to use AI-enabled tools req...",...,NaN,NaN,NaN,NaN,NaN,NaN,When I don’t trust AI’

***
### 2. Data Preprocessing Pipeline

Define and build a `ColumnTransformer` pipeline to handle missing values and encode categorical features.

In [48]:
# Features to impute with the median (numerical)
numerical_features = ['WorkExp', 'ToolCountWork','ToolCountPersonal', 'JobSat']

# Features to impute with the mode and One-Hot Encode (categorical/ordinal)
categorical_features = ['Age','EdLevel', 'DevType', 'ICorPM', 'RemoteWork','AIAgents','Country']

# A. Numerical Pipeline: Impute with Median
numerical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
])

# B. Categorical Pipeline: Impute with Mode, then One-Hot Encode
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore')) 
])
 
# C. Combine all steps using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_pipeline, numerical_features),
        ('cat', categorical_pipeline, categorical_features)
    ],
    remainder='drop' # Drop all other columns
)

#disabling this part so we can experiment on other models
#pipeline = Pipeline ([
#    ('preprocessor',preprocessor),
#    ('model', LinearRegression())
#])


# run the preprocessor on X
X_transformed = preprocessor.fit_transform(X)

print("Shape of processed data:", X_transformed.shape)
X_df = pd.DataFrame(X_transformed.toarray(), columns=preprocessor.get_feature_names_out())
X_df


Shape of processed data: (2343, 67)


,num__WorkExp,num__ToolCountWork,num__ToolCountPersonal,num__JobSat,cat__Age_18-24 years old,cat__Age_25-34 years old,cat__Age_35-44 years old,cat__Age_45-54 years old,cat__Age_55-64 years old,cat__Age_65 years or older,...,"cat__RemoteWork_Your choice (very flexible, you can come in when you want or just as needed)","cat__AIAgents_No, I use AI exclusively in copilot/autocomplete mode","cat__AIAgents_No, and I don't plan to","cat__AIAgents_No, but I plan to","cat__AIAgents_Yes, I use AI agents at work daily","cat__AIAgents_Yes, I use AI agents at work monthly or infrequently","cat__AIAgents_Yes, I use AI agents at work weekly",cat__Country_Austria,cat__Country_Germany,cat__Country_Switzerland
0,10.0,4.0,6.0,6.0,0.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
1,6.0,5.0,3.0,8.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
2,10.0,10.0,5.0,6.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,3.0,10.0,20.0,9.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,26.0,8.0,8.0,10.0,0.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2338,8.0,6.0,7.0,3.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2339,43.0,5.0,5.0,7.0,0.0,0.0,0.0,0.0,1.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2340,5.0,8.0,3.0,6.0,0.0,1.0,0.0,0.0,0.0,0.0,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2341,10.0,10.0,5.0,8.0,0.0,0.0,1.0,0.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0


***
### 3. Data Splitting

Split the processed data into training and testing sets.

In [44]:
X_train, X_test, y_train, y_test = train_test_split(X_transformed,y,test_size=0.2,random_state=42)
print(f"Train samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")

Train samples: 1874, Test samples: 469


***
### 4. Plotting Functions

Functions to help visualize the model's performance and structure.

***
### 5. Model Training and Evaluation

Train each of the three models, calculate accuracy, and generate plots.

#### 5.1 Base Model: Linear Regression

In [47]:
# Train the model
model = LinearRegression()
model.fit(X_train, y_train)
# Predict and Evaluate
prediction = model.predict(X_test)

#Evaluate Model Accuracy using root_mean_squared_error, r2_score, mean_absolute_error
mae = mean_absolute_error(y_test,prediction)
mse = mean_squared_error(y_test,prediction)
rmse = root_mean_squared_error(y_test,prediction)
r2 = r2_score(y_test,prediction) 

print("Mean Absolute Error: ",mae)
print("Mean Squared Error: ",mse)
print("Root Mean Squared Error: ", rmse)
print("r2 score:", r2)


Mean Absolute Error:  39625.399712866696
Mean Squared Error:  16302559582.401579
Root Mean Squared Error:  127681.47705286613
r2 score: -4.75830255113916
